# Exercise: Build a Fan-Out, Fan-In Workflow

Three Monte-Carlo jobs estimate pi from different random seeds at the same time, and a fourth job combines their counts. Your task is to build and run that graph with `dapi.workflows`. Pi makes the exercise self-checking, since the combined estimate must land near 3.14159.

```
pi-a ──┐
pi-b ──┼──> aggregate
pi-c ──┘
```

Work through the TODO cells. Each has a hint behind the fold, and the [solution notebook](DS_Pi_Fanout_Workflow.ipynb) shows one complete answer.

![Three quarter-circle panels, one per shard with its own random points, feeding the aggregation formula pi equals four times the summed inside counts over the summed samples.](pi_computation.png)

Each shard samples the **same quarter circle** with its own random seed, so the split is in the samples, not the domain. A point lands inside when $x^2 + y^2 \le 1$, each shard reports its count, and the aggregate sums them:

$$\pi \;\approx\; 4\,\frac{\sum_{\text{shards}} \text{inside}}{\sum_{\text{shards}} n}$$

In [ ]:
%pip install --quiet --upgrade dapi

**Restart the kernel once after the install**, then run from the next cell.

In [ ]:
from pathlib import Path

from dapi import DSClient

ds = DSClient()

# DesignSafe JupyterHub mounts your MyData at ~/MyData; scratch goes there
# since community folders are read-only. Anywhere else, write beside the
# notebook.
mydata = Path.home() / "MyData"
work_root = mydata if mydata.is_dir() else Path.cwd()

## The shard script (given)

Each shard samples points and writes how many fell inside the quarter circle. The seed and sample count arrive as job parameters, so one script serves all three shards.

In [ ]:
shard_src = """
import json
import os
import random

seed = int(os.environ["SEED"])
n = int(os.environ["N_SAMPLES"])
rng = random.Random(seed)
inside = sum(1 for _ in range(n) if rng.random() ** 2 + rng.random() ** 2 <= 1.0)
json.dump({"seed": seed, "n": n, "inside": inside}, open("pi_shard.json", "w"))
print(f"shard seed={seed}: {inside}/{n}")
"""
work_dir = work_root / "pi_fanout_scratch"
work_dir.mkdir(parents=True, exist_ok=True)
(work_dir / "shard_pi.py").write_text(shard_src)

shard_inputs = (
    "tapis://designsafe.storage.default/"
    + ds.tapis.username
    + "/dapi-workflows-demo/pi-inputs"
)
ds.files.upload(str(work_dir / "shard_pi.py"), shard_inputs + "/shard_pi.py")
print("shard script at", shard_inputs)

## TODO 1. The aggregator and the run root

The aggregator needs all three shard results, yet apps like `python-s3` accept exactly one input directory. Every task of a run archives under one **run root**, `MyData/dapi-workflows/<workflow>/<run_id>/<task>`, so point the aggregator's single input directory at the run root itself.

Write `aggregate_pi.py` so it reads every `<task>/inputDirectory/pi_shard.json` under its working directory, sums the counts, and writes `pi_estimate.json` with the combined estimate. Then choose a `run_id`, build the run-root URI for a workflow named `pi-demo`, and upload the script there.

<details><summary>Hint</summary>

From inside the staged run root, the shard results match the glob pattern `*/inputDirectory/pi_shard.json`. The estimate is `4 * inside / n` over the summed counts. Choosing `run_id = time.strftime("%Y%m%d-%H%M%S")` yourself, instead of letting `run()` default it, is what makes the run root a known path before the run.
</details>

Docs: [Parallel branches and fan-in](https://designsafe-ci.github.io/dapi/workflows#parallel-branches-and-fan-in).

In [ ]:
import time  # noqa: F401  (your answer uses it)

agg_src = """
# TODO: read every */inputDirectory/pi_shard.json, sum "inside" and "n",
# and write pi_estimate.json with keys pi_estimate and abs_error
"""
(work_dir / "aggregate_pi.py").write_text(agg_src)

run_id = ...  # TODO
run_root = (
    ...
)  # TODO: tapis://designsafe.storage.default/<you>/dapi-workflows/pi-demo/<run_id>
ds.files.upload(str(work_dir / "aggregate_pi.py"), run_root + "/aggregate_pi.py")

## TODO 2. Three shards, no edges between them

Generate a `python-s3` job for each shard, seeds 11, 22, and 33, two million samples each, one core, ten minutes, on `skx-dev`. Add each to the workflow with **no dependencies**, which is what lets the service run them at the same time. Give every job an `archiveFilter` so it archives only `pi_shard.json`.

<details><summary>Hint</summary>

`ds.jobs.generate(app_id="python-s3", input_dir_uri=shard_inputs, script_filename="shard_pi.py", ...)` with `extra_env_vars=[{"key": "SEED", ...}, {"key": "N_SAMPLES", ...}]`. The filter is `job["parameterSet"]["archiveFilter"] = {"includes": ["pi_shard.json", "**/pi_shard.json"], "includeLaunchFiles": False}`.
</details>

Docs: [Add the tasks to a workflow](https://designsafe-ci.github.io/dapi/workflows#step-2-add-the-tasks-to-a-workflow) and [Archive filters](https://designsafe-ci.github.io/dapi/workflows#archive-filters).

In [ ]:
allocation = "DS-Portal-SPARC2026"  # <-- replace with your allocation

from dapi.workflows import JobTask, Workflow  # noqa: F401  (your answer uses both)

wf = Workflow("pi-demo")
shard_ids = []
for tag, seed in (("pi-a", 11), ("pi-b", 22), ("pi-c", 33)):
    job = ...  # TODO: generate the shard job
    # TODO: name it, filter its archive, add it to the workflow
    shard_ids.append(tag)

## TODO 3. The fan-in

Generate the aggregator job. Its input directory is the run root you built in TODO 1, and its script is `aggregate_pi.py`. Add it with `depends_on` naming all three shards, so the service holds it until every shard has archived.

<details><summary>Hint</summary>

`wf.add(JobTask("aggregate", agg_job), depends_on=shard_ids)`. A plain URI carries no dependency information, so the explicit `depends_on` list is required here, unlike output references, which declare their own edges.
</details>

Docs: [Parallel branches and fan-in](https://designsafe-ci.github.io/dapi/workflows#parallel-branches-and-fan-in).

In [ ]:
agg_job = ...  # TODO
# TODO: add it to the workflow with its dependencies

wf.validate()
wf.visualize();

## TODO 4. Run it

Run the workflow with your `run_id` from TODO 1 (the aggregator's input points at that run's root, so the ids must match). Watch the stream, all three shards should go active together. Then download `pi_estimate.json` from the aggregator's archive and check the estimate.

<details><summary>Hint</summary>

`results = wf.run(ds, run_id=run_id, poll_interval=20)`. The report lands at `results["aggregate"]["archive_uri"] + "/inputDirectory/pi_estimate.json"`. Six million samples should land within about a thousandth of pi.
</details>

Docs: [Run and watch](https://designsafe-ci.github.io/dapi/workflows#step-5-run-and-watch).

In [ ]:
results = ...  # TODO

# TODO: download and print the estimate; assert abs_error < 5e-3

## Going further

Add a fourth shard without touching the aggregator script, or double `N_SAMPLES` and watch the error shrink by roughly the square root. For the sequential counterpart of this machinery, a sweep feeding a training job, see the [OpenSees ML workflow](../../opensees_ml/DS_OpenSees_ML_Workflow_DAG.ipynb).